# M1 Showcase — ERCOT HB_HOUSTON, June 2025

This notebook is a rough, visual proof of what the M1 ingestion pipeline actually produced — not a polished report (that's M5). Everything here reads from the DuckDB views the pipeline builds (`data/ercot.duckdb`), which are themselves views over the DQ-validated silver Parquet tables.

See [`docs/milestones/M1-skeleton-ingestion.md`](../docs/milestones/M1-skeleton-ingestion.md) for the full write-up.

**Contents:**
1. DAM vs RTM settlement point prices
2. Price volatility summary
3. Ancillary service clearing prices
4. System load
5. Data quality check results
6. A deliberately naive arbitrage number (⚠️ not the M2 optimizer — just a sense of scale)

In [1]:
import os
from pathlib import Path

import duckdb
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

# resolve to repo root regardless of whether this runs from notebooks/ (Jupyter Lab
# default) or the repo root (nbconvert default) so paths below are consistent either way
if not (Path.cwd() / "data" / "ercot.duckdb").exists():
    os.chdir(Path.cwd().parent)

con = duckdb.connect("data/ercot.duckdb", read_only=True)
con.execute("PRAGMA disable_progress_bar")
con.execute("SHOW TABLES").df()

,name
0,silver_dam_as_prices
1,silver_dam_spp
2,silver_load
3,silver_rtm_spp


## 1. DAM vs RTM settlement point prices — HB_HOUSTON

Day-ahead is the hourly-committed price; real-time is the 15-minute price that actual dispatch settles against. The gap between them is exactly what a DA-committed strategy is exposed to (M3), and the RTM series' volatility is where real-time arbitrage revenue comes from.

In [2]:
dam = con.execute(
    "SELECT interval_start, spp_usd_per_mwh FROM silver_dam_spp "
    "WHERE location = 'HB_HOUSTON' ORDER BY interval_start"
).df()
rtm = con.execute(
    "SELECT interval_start, spp_usd_per_mwh FROM silver_rtm_spp "
    "WHERE location = 'HB_HOUSTON' ORDER BY interval_start"
).df()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rtm["interval_start"], y=rtm["spp_usd_per_mwh"],
                          name="RTM (15-min)", line=dict(color="#636EFA", width=1)))
fig.add_trace(go.Scatter(x=dam["interval_start"], y=dam["spp_usd_per_mwh"],
                          name="DAM (hourly)", line=dict(color="#EF553B", width=1.5)))
fig.update_layout(
    title="HB_HOUSTON settlement point price — June 2025",
    xaxis_title="", yaxis_title="$/MWh", height=450, legend=dict(orientation="h", y=1.08),
)
fig.show()

In [3]:
# zoom in on a single day to see the intraday shape RTM volatility actually has
day = "2025-06-18"
dam_day = dam[dam["interval_start"].astype(str).str.startswith(day)]
rtm_day = rtm[rtm["interval_start"].astype(str).str.startswith(day)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=rtm_day["interval_start"], y=rtm_day["spp_usd_per_mwh"],
                          name="RTM (15-min)", line=dict(color="#636EFA", width=2), mode="lines+markers"))
fig.add_trace(go.Scatter(x=dam_day["interval_start"], y=dam_day["spp_usd_per_mwh"],
                          name="DAM (hourly)", line=dict(color="#EF553B", width=2, shape="hv")))
fig.update_layout(
    title=f"HB_HOUSTON intraday shape — {day}",
    xaxis_title="", yaxis_title="$/MWh", height=400, legend=dict(orientation="h", y=1.1),
)
fig.show()

## 2. Price volatility summary

RTM has a wider range than DAM by construction (it's uncommitted, 15-minute, and reacts to real-time grid conditions) — this spread is the raw material M2's perfect-foresight benchmark will price exactly, and M3's causal strategies will try to capture without hindsight.

In [4]:
summary = con.execute("""
    SELECT 'DAM' AS market, count(*) AS n_intervals,
           round(avg(spp_usd_per_mwh), 2) AS avg_usd_mwh,
           round(min(spp_usd_per_mwh), 2) AS min_usd_mwh,
           round(max(spp_usd_per_mwh), 2) AS max_usd_mwh,
           round(stddev(spp_usd_per_mwh), 2) AS stddev_usd_mwh
    FROM silver_dam_spp WHERE location = 'HB_HOUSTON'
    UNION ALL
    SELECT 'RTM', count(*), round(avg(spp_usd_per_mwh), 2), round(min(spp_usd_per_mwh), 2),
           round(max(spp_usd_per_mwh), 2), round(stddev(spp_usd_per_mwh), 2)
    FROM silver_rtm_spp WHERE location = 'HB_HOUSTON'
""").df()
summary

,market,n_intervals,avg_usd_mwh,min_usd_mwh,max_usd_mwh,stddev_usd_mwh
0,DAM,720,34.62,13.26,182.12,16.95
1,RTM,2880,31.94,-0.14,1250.73,33.89


## 3. Ancillary service clearing prices (DAM)

RegUp/RegDown/RRS/ECRS/NonSpin — capacity clearing prices from the Day-Ahead Market. This month is pre-RTC+B, so these are the *only* AS revenue stream available to a real-time resource; there's no real-time AS co-optimization yet (see [ADR 0003](../docs/adr/0003-rtcb-regime-as-first-class-dimension.md)).

In [5]:
as_prices = con.execute(
    "SELECT interval_start, product, mcpc_usd_per_mwh FROM silver_dam_as_prices ORDER BY interval_start"
).df()

fig = go.Figure()
for product in sorted(as_prices["product"].unique()):
    sub = as_prices[as_prices["product"] == product]
    fig.add_trace(go.Scatter(x=sub["interval_start"], y=sub["mcpc_usd_per_mwh"],
                              name=product, line=dict(width=1.2)))
fig.update_layout(
    title="DAM ancillary service clearing prices — June 2025 (ERCOT-wide)",
    xaxis_title="", yaxis_title="$/MW", height=450, legend=dict(orientation="h", y=1.08),
)
fig.show()

## 4. System load

ERCOT-wide load, for context — this is what's driving the price spikes above (summer demand).

In [6]:
load = con.execute(
    "SELECT interval_start, load_mw FROM silver_load WHERE zone = 'ERCOT' ORDER BY interval_start"
).df()

fig = go.Figure()
fig.add_trace(go.Scatter(x=load["interval_start"], y=load["load_mw"],
                          fill="tozeroy", line=dict(color="#00CC96", width=1)))
fig.update_layout(
    title="ERCOT system-wide load — June 2025",
    xaxis_title="", yaxis_title="MW", height=350, yaxis_range=[0, load["load_mw"].max() * 1.1],
)
fig.show()

## 5. Data quality checks

Re-running the actual `run_dq_checks` function from the pipeline (not reimplementing it here) against each silver table, to prove the CLEAN result reported in the M1 write-up.

In [7]:
import datetime as dt
import sys

sys.path.insert(0, "src")
from ercot_bess.transform.dq import run_dq_checks  # noqa: E402

DQ_SPEC = {
    "dam_spp": ("1h", 24),
    "rtm_spp": ("15m", 96),
    "load": ("1h", 24),
    "dam_as_prices": ("1h", 24),
}
start_date, end_date = dt.date(2025, 6, 1), dt.date(2025, 6, 30)

rows = []
for dataset, (freq, freq_per_day) in DQ_SPEC.items():
    df = con.execute(f"SELECT * FROM silver_{dataset}").pl()
    group_cols = [c for c in ("location", "product", "zone") if c in df.columns]
    tz = df.get_column("interval_start")[0].tzinfo
    start_dt = dt.datetime.combine(start_date, dt.time.min, tzinfo=tz)
    end_dt = dt.datetime.combine(end_date + dt.timedelta(days=1), dt.time.min, tzinfo=tz)
    report = run_dq_checks(df, dataset, "interval_start", start_dt, end_dt, freq, freq_per_day, group_cols)
    rows.append({
        "dataset": dataset,
        "status": "CLEAN" if report.is_clean else "ISSUES",
        "missing_intervals": len(report.missing_intervals),
        "duplicate_timestamps": len(report.duplicate_timestamps),
        "dst_issues": len(report.dst_day_issues),
    })

import pandas as pd
pd.DataFrame(rows)

,dataset,status,missing_intervals,duplicate_timestamps,dst_issues
0,dam_spp,CLEAN,0,0,0
1,rtm_spp,CLEAN,0,0,0
2,load,CLEAN,0,0,0
3,dam_as_prices,CLEAN,0,0,0


## 6. Rough arbitrage sense-of-scale (⚠️ not the M2 optimizer)

This is deliberately naive and **not** the perfect-foresight LP benchmark M2 will build. It ignores round-trip efficiency, degradation cost, the daily cycle limit, and multi-day state-of-charge carryover — it just asks, per day: if a 100MW/200MWh battery charged for 2 hours at that day's 8 cheapest RTM intervals and discharged for 2 hours at that day's 8 priciest intervals, what would that be worth? It's a lower bound on what M2's real LP will find (the LP can also work overnight boundaries, partial-power dispatch, and AS capacity revenue), but it's enough to show the data actually contains real, capturable spread — which is the entire premise of this project.

In [8]:
from ercot_bess.models.battery import BatterySpec  # noqa: E402

spec = BatterySpec()  # 100MW / 200MWh defaults
intervals_to_fill = int(spec.energy_mwh / spec.power_mw / 0.25)  # 15-min intervals to charge/discharge fully
mwh_per_interval = spec.power_mw * 0.25

rtm["date"] = rtm["interval_start"].dt.date
daily_rows = []
for date, day_df in rtm.groupby("date"):
    prices = day_df["spp_usd_per_mwh"].sort_values()
    if len(prices) < 2 * intervals_to_fill:
        continue
    charge_cost = prices.iloc[:intervals_to_fill].sum() * mwh_per_interval
    discharge_revenue = prices.iloc[-intervals_to_fill:].sum() * mwh_per_interval
    daily_rows.append({"date": date, "naive_daily_revenue_usd": discharge_revenue - charge_cost})

naive = pd.DataFrame(daily_rows)
total = naive["naive_daily_revenue_usd"].sum()
print(f"Naive single-cycle-per-day arbitrage, June 2025, HB_HOUSTON: ${total:,.0f}")
print(f"Average per day: ${naive['naive_daily_revenue_usd'].mean():,.0f}")

fig = go.Figure(go.Bar(x=naive["date"].astype(str), y=naive["naive_daily_revenue_usd"], marker_color="#AB63FA"))
fig.update_layout(title="Naive daily arbitrage revenue (illustrative only)", yaxis_title="$", height=350)
fig.show()

Naive single-cycle-per-day arbitrage, June 2025, HB_HOUSTON: $391,354
Average per day: $13,045


## Next: M2

The real number — % of perfect-foresight revenue captured — needs the actual CVXPY/HiGHS LP with efficiency, SoC dynamics, degradation cost, and cycle limits enforced properly, plus AS capacity co-optimization. That's M2.